# GHARIBO — model-load stability preflight (DEC-0041)

| Field | Value |
|-------|-------|
| Authorization | `DEC-0041` — NO-INFERENCE PREFLIGHT |
| Attempt #5 | NOT AUTHORIZED |
| Base arm | `unsloth/gpt-oss-20b` (unadapted) |
| Immutable distribution revision | `093fba6992ef5a7152481afec0bdfca1ac486998` |

**This notebook does NOT run evaluation. It does NOT attach TEST data, prompts, gold,
adapters, or any inference primitive. It proves ONLY that the production loader installs,
loads tokenizer, loads BASE, and resolves the distribution to the immutable revision.**

The install and model-load code is extracted verbatim from the production evaluation
kernel (build-eval-kernel.mjs). The preflight and the production eval call the SAME
implementation — no copy-pasted duplicate loader logic.

In [ ]:
# --- governed preflight pins (injected by the generator; do not hand-edit) ---
# The pins arrive as an EMBEDDED JSON STRING and are decoded with json.loads().
_PINS_JSON = r'''{
    "attempt5Authorized": false,
    "authorizationDecisionId": "DEC-0041",
    "baseModel": "openai/gpt-oss-20b",
    "baseModelRevision": "6cee5e81ee83917806bbde320786a8fb61efebee",
    "decision": "NO_INFERENCE_PREFLIGHT_AUTHORIZATION",
    "engineDependencies": [
        {
            "name": "unsloth",
            "spec": "unsloth==2026.9.4"
        },
        {
            "name": "unsloth_zoo",
            "spec": "unsloth_zoo==2026.9.3"
        },
        {
            "name": "transformers",
            "spec": "transformers==4.56.2"
        },
        {
            "name": "peft",
            "spec": "peft==0.20.0"
        },
        {
            "name": "trl",
            "spec": "trl==0.22.2"
        },
        {
            "name": "datasets",
            "spec": "datasets==5.0.1"
        },
        {
            "name": "accelerate",
            "spec": "accelerate==1.15.0"
        },
        {
            "name": "bitsandbytes",
            "spec": "bitsandbytes==0.50.2"
        },
        {
            "name": "openai-harmony",
            "spec": "openai-harmony==0.0.8"
        }
    ],
    "engineFreeze": "unsloth-freeze-2026.09.15",
    "frozenNoDeps": [
        "unsloth",
        "unsloth_zoo"
    ],
    "immutableDistributionRevision": "093fba6992ef5a7152481afec0bdfca1ac486998",
    "loaderModelId": "unsloth/gpt-oss-20b",
    "maxSeqLength": 1024,
    "maximumKernelPushes": 1,
    "preservedCandidates": [
        "torch",
        "triton"
    ],
    "skipWhenPreserved": [
        "triton_kernels"
    ],
    "supportNoDeps": [
        "torchao>=0.16.0"
    ]
}'''

import hashlib, json, os, sys, time

PINS = json.loads(_PINS_JSON)

# The pins must survive the round-trip EXACTLY.
assert json.loads(json.dumps(PINS, sort_keys=True)) == PINS, 'pin round-trip is not stable'

# Preflight pins must NOT carry evaluation or TEST material.
_FORBIDDEN_PREFLIGHT_KEYS = [
    'testSplitHash', 'testRecordCount', 'datasetHash',
    'candidateAdapterSha256', 'candidateArmId',
    'temperature', 'doSample', 'topP', 'topK', 'maxNewTokens', 'seed', 'repeats',
]
for _k in _FORBIDDEN_PREFLIGHT_KEYS:
    assert _k not in PINS, f'PREFLIGHT REFUSES: evaluation pin {_k!r} is present in a no-inference kernel'

assert PINS['authorizationDecisionId'] == 'DEC-0041', 'unexpected authorization'
assert PINS['decision'] == 'NO_INFERENCE_PREFLIGHT_AUTHORIZATION', 'unexpected decision scope'
assert PINS['maximumKernelPushes'] == 1, 'preflight is bounded to ONE kernel push'
assert PINS['attempt5Authorized'] is False, 'ATTEMPT #5 IS NOT AUTHORIZED'
assert PINS['immutableDistributionRevision'] == '093fba6992ef5a7152481afec0bdfca1ac486998', \
    'immutable distribution revision must be the DEC-0037 proven SHA'

print('authorization :', PINS['authorizationDecisionId'], '-', PINS['decision'])
print('attempt #5   : NOT AUTHORIZED')

In [ ]:
# --- environment / GPU (recorded, never assumed) ---
ENV = {
    'python': sys.version.split()[0],
    'cuda_available': None,
    'gpu_name': None,
    'started_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
try:
    import torch
    ENV['torch'] = torch.__version__
    ENV['cuda_available'] = bool(torch.cuda.is_available())
    ENV['cuda'] = torch.version.cuda
    if torch.cuda.is_available():
        ENV['gpu_name'] = torch.cuda.get_device_name(0)
        ENV['gpu_count'] = torch.cuda.device_count()
        ENV['compute_capability'] = '.'.join(map(str, torch.cuda.get_device_capability(0)))
except Exception as exc:
    ENV['torch'] = None
    ENV['torch_error'] = repr(exc)

print(json.dumps(ENV, indent=2))
assert ENV.get('cuda_available'), 'this preflight requires a GPU'

In [ ]:
# --- input isolation: NO TEST data, NO prompts, NO gold, NO adapter ---
# The preflight must be structurally incapable of accessing evaluation material.
_input_dir = '/kaggle/input'
_has_input = os.path.isdir(_input_dir) and any(os.scandir(_input_dir))
if _has_input:
    _entries = sorted(os.listdir(_input_dir))
    print('WARNING: /kaggle/input is not empty:', _entries)
    # Check for forbidden payloads
    _forbidden_names = {'prompts.jsonl', 'gold.jsonl', 'test.jsonl', 'train.jsonl', 'validation.jsonl',
                         'adapter_config.json', 'adapter_model.safetensors'}
    for _root, _dirs, _files in os.walk(_input_dir):
        for _f in _files:
            assert _f not in _forbidden_names, \
                f'PREFLIGHT REFUSES: forbidden file {_f!r} found in /kaggle/input'
else:
    print('input isolation: /kaggle/input is empty or absent')
print('PREFLIGHT_TEST_ACCESS_NO')

In [ ]:
# --- preventive hardening: HF_HUB_DISABLE_XET=1 + pre-download at immutable revision ---
#
# DEC-0040 recorded a Xet transport warning immediately before the attempt-#4 failure.
# Causation is UNPROVEN (ordering != proof), but the transport fallback path is the one
# observable difference between the PASS (DEC-0037, immutable SHA) and the FAIL (attempt #4,
# `main`). Disabling Xet removes the fallback path from the critical path entirely, and
# pre-downloading the distribution at the pinned revision forces the HF cache to hold the
# immutable SHA. The loader then resolves from cache rather than from a live ref.
#
# This is PREVENTIVE HARDENING, not a causal claim. It is recorded explicitly so future
# evaluation runs must use the exact same path.
os.environ['HF_HUB_DISABLE_XET'] = '1'
print('HF_HUB_DISABLE_XET=1 (preventive hardening, recorded)')

# Pre-download the distribution at the immutable revision.
# This forces the HF cache to hold the pinned SHA so the loader cannot resolve to `main`.
from huggingface_hub import snapshot_download

_dist_id = PINS['loaderModelId'] + '-unsloth-bnb-4bit'
_dist_rev = PINS['immutableDistributionRevision']
print(f'pre-downloading distribution: {_dist_id} @ {_dist_rev}')
snapshot_download(repo_id=_dist_id, revision=_dist_rev)
print('distribution pre-downloaded at immutable revision')

In [ ]:
# --- dependencies: the SAME governed engine stack the training run used ---
#
# Reproduces the accepted three-stage `uv` discipline (engine freeze
# unsloth-freeze-2026.09.15). The ad-hoc %pip sequence this replaced was DEFECT 1 of the
# pre-execution blocker recorded in DEC-0033 / BLK-0004: it submitted the whole
# frozen set to ONE resolver transaction, which is unsatisfiable because
# unsloth/unsloth_zoo cap `datasets<4.4.0` while the freeze pins `datasets==5.0.1`.
#
# Guarantees:
#   1. No -qqq on install commands - complete stdout + stderr are captured.
#   2. Every stage is dry-run with its EXACT arguments immediately before it runs.
#   3. On failure a bounded redacted diagnostic is persisted, and the raised error
#      carries the real resolver/package reason (never just 'exit code 1').
#   4. Preinstalled torch/triton are preserved and constraint-pinned, so no stage
#      can upgrade them off the +cu128 build Kaggle provides.
#   5. triton_kernels is skipped on the Kaggle preserve path (upstream-aligned).
import importlib.metadata as _metadata
import os, pathlib, re, shutil, subprocess, sys

WORKING = pathlib.Path('/kaggle/working')
if not WORKING.exists():
    WORKING = pathlib.Path('.')

INSTALL_DIAGNOSTIC_PATH = WORKING / 'eval-install-diagnostic.json'

def redact(text):
    # Removes anything credential-shaped before it is printed or persisted.
    text = re.sub(r'hf_[A-Za-z0-9]{10,}', '<redacted-hf-token>', text)
    text = re.sub(r'(?i)(api[_-]?key|token|secret|password)([\"\']?\s*[:=]\s*)([^\s\"\',]+)',
                  r'\1\2<redacted>', text)
    return text

def scrub_paths(text):
    return text.replace('/kaggle/input/', '/kaggle/input/<dataset>/')

def run_install_command(cmd, phase, timeout=None):
    display = scrub_paths(redact(' '.join(cmd)))
    print('$', display)
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    if proc.returncode != 0:
        max_diag = 32000
        out_trim = scrub_paths(redact(proc.stdout or ''))[-max_diag:]
        err_trim = scrub_paths(redact(proc.stderr or ''))[-max_diag:]
        diagnostic = {'phase': phase, 'command': display, 'exit_code': proc.returncode,
                      'stdout_redacted': out_trim, 'stderr_redacted': err_trim}
        try:
            INSTALL_DIAGNOSTIC_PATH.write_text(json.dumps(diagnostic, indent=1), encoding='utf-8')
        except Exception as exc:
            print('could not persist install diagnostic:', type(exc).__name__)
        raise RuntimeError(
            '%s failed (exit code %d).\n--- redacted stdout (last %d chars) ---\n%s\n'
            '--- redacted stderr (last %d chars) ---\n%s\nDiagnostic persisted to %s'
            % (phase, proc.returncode, len(out_trim), out_trim, len(err_trim), err_trim,
               INSTALL_DIAGNOSTIC_PATH.name))
    return proc

print('Bootstrapping uv...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', '-qqq', 'uv'], check=True)
UV = shutil.which('uv') or os.path.join(os.path.dirname(sys.executable), 'uv')
if not shutil.which('uv') and not os.path.exists(UV):
    raise RuntimeError('uv was installed but is not on PATH - cannot continue.')

if os.environ.get('VIRTUAL_ENV'):
    TARGET_FLAGS = ['--python', sys.executable]
else:
    TARGET_FLAGS = ['--system', '--python', sys.executable]

# Turing-only build target: keeps any source build from emitting sm_80+ kernels a
# T4 cannot load.
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'
os.environ.setdefault('CMAKE_CUDA_ARCHITECTURES', '75')

ON_KAGGLE = os.path.isdir('/kaggle')

def module_present(modname):
    try:
        __import__(modname)
        return True
    except Exception:
        return False

PRESERVED = {}
if ON_KAGGLE:
    for _name in PINS['preservedCandidates']:
        if module_present(_name):
            PRESERVED[_name] = _metadata.version(_name)

SKIPPED = set()
if PRESERVED:
    SKIPPED.update(PINS['skipWhenPreserved'])

resolver_specs = []
frozen_specs = []
for _dep in PINS['engineDependencies']:
    if _dep['name'] in PRESERVED:
        print('preserving preinstalled %s==%s (not re-resolved against PyPI)'
              % (_dep['name'], PRESERVED[_dep['name']]))
        continue
    if _dep['name'] in SKIPPED:
        print('skipping %s on the Kaggle preserve path (upstream-aligned)' % _dep['name'])
        continue
    if _dep['name'] in PINS['frozenNoDeps']:
        frozen_specs.append(_dep['spec'])
    else:
        resolver_specs.append(_dep['spec'])

print('Stage 1 (resolver-managed):', resolver_specs)
print('Stage 2 (--no-deps frozen set):', frozen_specs)
print('Stage 3 (--no-deps support):', PINS['supportNoDeps'])

# Exact constraints stop any transitive dependency from upgrading the preserved builds.
CONSTRAINT_PATH = WORKING / 'eval-preserved-constraints.txt'
CONSTRAINT_PATH.write_text(''.join('%s==%s\n' % _i for _i in sorted(PRESERVED.items())),
                           encoding='utf-8')
CONSTRAINT_FLAGS = ['--constraint', str(CONSTRAINT_PATH)] if PRESERVED else []

BASE = [UV, 'pip', 'install', *TARGET_FLAGS, '--no-cache-dir', *CONSTRAINT_FLAGS]

INSTALL_PLAN = [
    ('install', [*BASE, *resolver_specs]),
    ('frozen-no-deps', [*BASE, '--upgrade', '--no-deps', *frozen_specs]),
    ('support-no-deps', [*BASE, '--no-deps', '--upgrade', *PINS['supportNoDeps']]),
]

for _phase, _cmd in INSTALL_PLAN:
    run_install_command([*_cmd, '--dry-run'], _phase + '-dry-run')
    run_install_command(_cmd, _phase)

for _name, _version in PRESERVED.items():
    _actual = _metadata.version(_name)
    if _actual != _version:
        raise RuntimeError('preserved dependency changed: %s==%s -> %s'
                           % (_name, _version, _actual))

print('install complete')
print('preserved:', PRESERVED or 'none')
print('skipped:', sorted(SKIPPED) or 'none')

In [ ]:
# --- enforce the pinned BASE revision against the LIVE base repo ---
#
# The loader call deliberately does NOT take a revision (see the BASE-load cell: passing one
# to the Unsloth distribution id makes the load fail). That makes this cell LOAD-BEARING:
# without it, nothing in this notebook would tie the measured base to the accepted revision,
# and the arm could silently evaluate a different snapshot of the base model.
#
# The revision is resolved from the REAL base repo (`openai/gpt-oss-20b`), never guessed and
# never taken from a cached value, and a mismatch aborts the run.
from huggingface_hub import HfApi

def resolve_base_revision(repo_id):
    info = HfApi().model_info(repo_id=repo_id)
    sha = getattr(info, 'sha', None)
    if not isinstance(sha, str) or len(sha) != 40:
        raise RuntimeError(f'huggingface_hub reported no 40-hex revision for {repo_id!r}: {sha!r}')
    return sha

ENV['base_repo'] = PINS['baseModel']
BASE_MODEL_REVISION = resolve_base_revision(PINS['baseModel'])
ENV['base_model_revision_resolved'] = BASE_MODEL_REVISION
print('base repo            :', PINS['baseModel'])
print('base revision (live) :', BASE_MODEL_REVISION)
print('base revision (pinned):', PINS['baseModelRevision'])
assert BASE_MODEL_REVISION == PINS['baseModelRevision'], (
    f"REFUSING TO RUN: base revision drifted. pinned {PINS['baseModelRevision']}, "
    f"live {BASE_MODEL_REVISION}"
)
print('base revision pin    : ENFORCED against the live repo')

In [ ]:
# --- load BASE, unadapted ---
#
# This is the SAME FastLanguageModel.from_pretrained call the production eval kernel uses,
# with NO revision= argument (the governed convention proven by DEC-0037). The pre-download
# above has already placed the immutable revision in the HF cache, so the loader resolves
# from cache at the pinned SHA rather than from a live ref that may fall back to `main`.
#
# The pinned base revision is asserted against the LIVE base repo in the cell above, and
# the distribution revision is asserted below after load.
from unsloth import FastLanguageModel
import torch

base_model, base_tok = FastLanguageModel.from_pretrained(
    model_name=PINS['loaderModelId'],
    max_seq_length=PINS['maxSeqLength'],
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)
base_model.eval()

print('PREFLIGHT_INSTALL_PASS')
print('PREFLIGHT_TOKENIZER_PASS')
print('PREFLIGHT_MODEL_LOAD_PASS')

# --- identity / revision assertions ---
LOADER_NAME = getattr(getattr(base_model, 'config', None), '_name_or_path', None)
print('base loaded from:', LOADER_NAME)
assert LOADER_NAME is not None, 'PREFLIGHT FAIL: could not read the loaded model identity'
assert 'gpt-oss-20b' in str(LOADER_NAME), \
    f'PREFLIGHT FAIL: loader resolved to {LOADER_NAME!r}, which is not the accepted base'

# The distribution revision must be the immutable SHA, NOT `main`.
# This is the TARGET INVARIANT: FAIL CLOSED if observed revision is anything else.
_loaded_revision = getattr(getattr(base_model, 'config', None), '_commit_hash', None)
_tok_id = str(getattr(base_tok, 'name_or_path', ''))
print(f'OBSERVED_DISTRIBUTION_ID={_tok_id}')
print(f'OBSERVED_DISTRIBUTION_REVISION={_loaded_revision}')

assert _loaded_revision is not None, 'PREFLIGHT FAIL: no _commit_hash on loaded model config'
assert _loaded_revision == PINS['immutableDistributionRevision'], \
    f'PREFLIGHT FAIL: resolved revision {_loaded_revision!r} != immutable SHA {PINS["immutableDistributionRevision"]!r}'
assert _loaded_revision != 'main', \
    'PREFLIGHT FAIL: resolved to mutable ref "main" — this is the attempt-#4 failure mode'

print('PREFLIGHT_DISTRIBUTION_ID_PASS')
print('PREFLIGHT_DISTRIBUTION_REVISION_PASS')
print('PREFLIGHT_BASE_REVISION_PASS')

# --- gradient check (no training) ---
_grad_enabled = any(p.requires_grad for p in base_model.parameters())
assert _grad_enabled is False, 'PREFLIGHT FAIL: gradients are enabled on BASE'

# --- no inference, no generation ---
# Structural: this notebook contains no model.generate() call, no prompts, no gold.
print('PREFLIGHT_TEST_ACCESS_NO')
print('PREFLIGHT_INFERENCE_NO')
print('PREFLIGHT_COMPLETE')